In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

FILE = Path("../../scratch/fsm2/S01/2020_07/driving_data.nc")

with xr.open_dataset(FILE) as source:
    ds = source.load()

display(ds)

# Variable dimensions, units and missing values.
display(pd.DataFrame([
    {
        "variable": name,
        "dimensions": ", ".join(var.dims),
        "shape": str(var.shape),
        "dtype": str(var.dtype),
        "units": var.attrs.get("units", ""),
        "missing (%)": round(float(var.isnull().mean()) * 100, 3),
    }
    for name, var in ds.data_vars.items()
]).set_index("variable"))

# Check the time axis.
time_name = "valid_time" if "valid_time" in ds.coords else "time"
times = pd.DatetimeIndex(ds[time_name].values)
print(f"Time range: {times.min()} to {times.max()}")
print(f"Number of timestamps: {len(times)}")
print(f"Unique timestamps: {times.is_unique}")
print(f"Strictly hourly: {bool(np.all(np.diff(times.asi8) == pd.Timedelta(hours=1).value))}")

# Show the first 10 hours at the central grid point.
point = ds.isel(
    latitude=ds.sizes["latitude"] // 2,
    longitude=ds.sizes["longitude"] // 2,
)
print(f"\nSample point: {float(point.latitude):.3f}°, {float(point.longitude):.3f}°")
hourly_variables = [name for name, var in point.data_vars.items() if time_name in var.dims]
display(point[hourly_variables].isel({time_name: slice(0, 10)}).to_dataframe())

print("\nStatic vegetation at this point:")
display(point[["VAI", "vegh"]].to_array().to_series())

<xarray.Dataset> Size: 92MB
Dimensions:     (valid_time: 744, latitude: 38, longitude: 81)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 6kB 2020-07-01 ... 2020-07-31T23:...
  * latitude    (latitude) float64 304B 64.8 64.7 64.6 64.5 ... 61.3 61.2 61.1
  * longitude   (longitude) float64 648B -126.6 -126.5 -126.4 ... -118.7 -118.6
Data variables:
    d2m         (valid_time, latitude, longitude) float32 9MB 286.3 ... 284.4
    t2m         (valid_time, latitude, longitude) float32 9MB 290.2 ... 302.0
    sf          (valid_time, latitude, longitude) float32 9MB 0.0 0.0 ... 0.0
    ssrd        (valid_time, latitude, longitude) float32 9MB 2.052e+07 ... 2...
    strd        (valid_time, latitude, longitude) float32 9MB 2.834e+07 ... 2...
    u10         (valid_time, latitude, longitude) float32 9MB 1.32 ... -0.04897
    v10         (valid_time, latitude, longitude) float32 9MB -1.888 ... 1.957
    sp          (valid_time, latitude, longitude) float32 9MB 9.688e+04 ... 9...
    tp          (valid_time, latitude, longitude) float32 9MB 0.00698 ... 4.2...
    w10         (valid_time, latitude, longitude) float32 9MB 2.303 ... 1.957
    VAI         (latitude, longitude) float32 12kB 1.871 1.855 ... 2.14 1.935
    vegh        (latitude, longitude) float32 12kB 5.32 5.042 ... 7.134 6.5
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-11-04T13:35 GRIB to CDM+CF via cfgrib-0.9.1...

,dimensions,shape,dtype,units,missing (%)
variable,,,,,
d2m,"valid_time, latitude, longitude","(744, 38, 81)",float32,K,0.00
t2m,"valid_time, latitude, longitude","(744, 38, 81)",float32,K,0.00
sf,"valid_time, latitude, longitude","(744, 38, 81)",float32,m of water equivalent,0.00
ssrd,"valid_time, latitude, longitude","(744, 38, 81)",float32,J m**-2,0.00
strd,"valid_time, latitude, longitude","(744, 38, 81)",float32,J m**-2,0.00
u10,"valid_time, latitude, longitude","(744, 38, 81)",float32,m s**-1,0.00
v10,"valid_time, latitude, longitude","(744, 38, 81)",float32,m s**-1,0.00
sp,"valid_time, latitude, longitude","(744, 38, 81)",float32,Pa,0.00
tp,"valid_time, latitude, longitude","(744, 38, 81)",float32,m,0.00


Time range: 2020-07-01 00:00:00 to 2020-07-31 23:00:00
Number of timestamps: 744
Unique timestamps: True
Strictly hourly: True

Sample point: 62.900°, -122.600°


,d2m,t2m,sf,ssrd,strd,u10,v10,sp,tp,w10,latitude,longitude
valid_time,,,,,,,,,,,,
2020-07-01 00:00:00,283.214600,295.280029,0.0,23284524.0,28362596.00,-1.992966,1.484314,96004.1250,0.000272,2.484975,62.9,-122.6
2020-07-01 01:00:00,283.117188,294.612061,0.0,1415442.0,1270046.75,-2.615128,2.086288,96034.5625,0.000050,3.345369,62.9,-122.6
2020-07-01 02:00:00,284.451416,292.730713,0.0,2261190.0,2567790.75,-1.853928,1.521240,96100.8125,0.000292,2.398170,62.9,-122.6
2020-07-01 03:00:00,284.921875,292.375488,0.0,2814201.0,3854244.25,-0.947205,1.083099,96102.1875,0.000292,1.438854,62.9,-122.6
2020-07-01 04:00:00,285.944336,291.908691,0.0,3165580.0,5101897.00,-0.878647,0.615234,96118.6875,0.000292,1.072629,62.9,-122.6
2020-07-01 05:00:00,286.218262,291.172363,0.0,3361515.0,6322951.00,-1.252625,0.414032,96143.4375,0.000292,1.319277,62.9,-122.6
2020-07-01 06:00:00,286.065674,290.028564,0.0,3415102.0,7505681.00,-1.438187,0.282104,96164.9375,0.000292,1.465593,62.9,-122.6
2020-07-01 07:00:00,286.389893,289.099854,0.0,3415122.0,8721027.00,-1.154205,0.085236,96225.3750,0.000292,1.157348,62.9,-122.6
2020-07-01 08:00:00,285.791260,288.165527,0.0,3415122.0,9923662.00,-1.169037,-0.472366,96241.5000,0.000292,1.260864,62.9,-122.6



Static vegetation at this point:


variable
VAI     1.697661
vegh    7.275062
dtype: float32